# Malware Analysis with YARA

> **Authorized lab use only.** This notebook is an educational demonstration. Run it only with synthetic data or systems you own and are permitted to test.


In [ ]:
pip install yara-python

In [ ]:
## YARA MALWARE ANALYSIS Completed on 4-1-25
import os
import yara

def create_yara_rule(rule_name, rule_content, rule_directory="./yara_rules"):
    """Creates a YARA rule file dynamically."""
    os.makedirs(rule_directory, exist_ok=True)
    rule_path = os.path.join(rule_directory, f"{rule_name}.yara")
    with open(rule_path, "w") as rule_file:
        rule_file.write(rule_content)
    print(f"YARA rule saved at: {rule_path}")
    return rule_path

def compile_yara_rules(rule_directory="./yara_rules"):
    """Compiles all YARA rules from a given directory."""
    rule_files = {f: os.path.join(rule_directory, f) for f in os.listdir(rule_directory) if f.endswith(".yar") or f.endswith(".yara")}
    if not rule_files:
        raise ValueError("No YARA rules found in the specified directory.")
    rules = yara.compile(filepaths=rule_files)
    print("YARA rules compiled successfully!")
    return rules

def scan_file_with_yara(rules, file_path):
    """Scans a file using compiled YARA rules and returns matches."""
    if not os.path.exists(file_path):
        raise FileNotFoundError(f"File not found: {file_path}")
    matches = rules.match(file_path)
    if matches:
        print(f"Malware signatures detected in {file_path}:")
        for match in matches:
            print(f"- {match.rule}")
    else:
        print(f"No malware detected in {file_path}.")

def create_test_file(file_path):
    """Creates a test file with a known malware signature."""
    with open(file_path, "w") as f:
        f.write("malware_signature")  # This matches the YARA rule
    print(f"Test file created: {file_path}")

def main():
    rule_directory = "./yara_rules"
    file_to_scan = "./suspicious_file.exe"

    yara_rule_content = """
    rule Suspicious_Malware {
        meta:
            description = "Detects a suspicious malware signature"
            author = "Automated Script"
            date = "2025-04-01"
        strings:
            $malicious_string = "malware_signature"
            $hex_pattern = { E8 ?? ?? ?? ?? 5D C3 }
        condition:
            any of them
    }
    """

    try:
        print("Creating YARA rule...")
        create_yara_rule("malware_rule", yara_rule_content, rule_directory)

        print("Compiling YARA rules...")
        yara_rules = compile_yara_rules(rule_directory)

        print("Creating test file...")
        create_test_file(file_to_scan)

        print("Scanning file for malware...")
        scan_file_with_yara(yara_rules, file_to_scan)

    except Exception as e:
        print(f"Error: {e}")

if __name__ == "__main__":
    main()